# UPI Fraud Rings: A100 training driver
Set REPO_URL, BRANCH, and optionally COMMIT_SHA. Keep credentials in Colab Secrets.

In [ ]:
!nvidia-smi -L
!nproc
!free -h

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/zahy294/upi-fraud-ml.git'
BRANCH = 'codex/target-070'
COMMIT_SHA = ''  # fill with agreed full SHA

In [ ]:
!git clone --branch {BRANCH} --single-branch {REPO_URL} /content/upi-fraud-ml
%cd /content/upi-fraud-ml
if COMMIT_SHA:
    !git checkout {COMMIT_SHA}
!git rev-parse HEAD
!python -m pip install -q -r requirements.txt

Put the five CSVs in /content/drive/MyDrive/upi-fraud/data/raw.

In [ ]:
!mkdir -p data/raw
!cp /content/drive/MyDrive/upi-fraud/data/raw/*.csv data/raw/
!python run.py audit --config configs/xgb_a100.yaml
!python -m unittest discover -s tests -v
!python run.py smoke --config configs/xgb_a100.yaml

In [ ]:
!python run.py features --config configs/xgb_a100.yaml
!python run.py train --config configs/xgb_a100.yaml

Inspect manifest.json. Only after accepting validation, run predict and copy the complete run to Drive.

In [ ]:
RUN_DIR = 'runs/<validated-run>'
!python run.py predict --config configs/xgb_a100.yaml --run-dir {RUN_DIR}
!mkdir -p /content/drive/MyDrive/upi-fraud/runs
!cp -r {RUN_DIR} /content/drive/MyDrive/upi-fraud/runs/
!cp submissions/xgb_a100.csv /content/drive/MyDrive/upi-fraud/runs/